# boe-extractor en Colab (GPU T4)

Notebook fino: toda la lógica está en el paquete. Antes, en tu PC: `uv run boe-sft` y sube `data/sft/` a `MyDrive/boe-extractor/sft/`.

Tres partes, **cada una en un entorno nuevo** (Entorno de ejecución → Desconectar y eliminar), porque vLLM y Unsloth piden versiones distintas de torch.

Al acabar, baja `MyDrive/boe-extractor/preds/*.jsonl` a `data/preds/` y en tu PC: `uv run --group exp boe-eval data/preds/<fichero>.jsonl`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/boe-extractor'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Modelo base sin ajustar (vLLM + decodificación restringida)

In [ ]:
!git clone -q https://github.com/inigo99/boe-extractor && pip install -q -e boe-extractor vllm
!python -m boe_extractor.infer --datos /content/drive/MyDrive/boe-extractor/sft/gold.jsonl --salida /content/drive/MyDrive/boe-extractor/preds/qwen2.5-1.5b-base.jsonl

## 2. QLoRA (Unsloth) — entorno nuevo

Unas horas en T4. Si Colab corta la sesión, vuelve a ejecutar: sigue desde el último checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone -q https://github.com/inigo99/boe-extractor && pip install -q -e boe-extractor unsloth
!python -m boe_extractor.train.qlora --datos /content/drive/MyDrive/boe-extractor/sft --salida /content/drive/MyDrive/boe-extractor/qlora-v1

## 3. Modelo ajustado (vLLM + adaptador) — entorno nuevo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone -q https://github.com/inigo99/boe-extractor && pip install -q -e boe-extractor vllm
!python -m boe_extractor.infer --datos /content/drive/MyDrive/boe-extractor/sft/gold.jsonl --lora /content/drive/MyDrive/boe-extractor/qlora-v1/adaptador --salida /content/drive/MyDrive/boe-extractor/preds/qwen2.5-1.5b-qlora-v1.jsonl